In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
!unzip "/content/drive/MyDrive/manifest-1786753014866.zip" -d "/content/drive/MyDrive/"

Archive:  /content/drive/MyDrive/manifest-1786753014866.zip
replace /content/drive/MyDrive/__MACOSX/._manifest-1786753014866? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/drive/MyDrive/__MACOSX/manifest-1786753014866/._LIDC-IDRI? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os

In [ ]:
colab_path="/content/drive/MyDrive/manifest-1786753014866/LIDC-IDRI"

In [ ]:
with open('/root/.pylidcrc','w') as f:
  f.write("[dicom]\n")
  f.write(f"path={colab_path}\n")

In [ ]:
print("Config file created successfully!")

Config file created successfully!


In [ ]:
!pip install pylidc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.4 MB/s eta 0:00:00


In [ ]:
import pylidc as pl
import matplotlib.pyplot as plt


#1. QUERY THE DATABSE FOR FIRST SCAN

In [ ]:
scan = pl.query(pl.Scan).first()
print(f"Loaded Patient ID: {scan.patient_id}")

Loaded Patient ID: LIDC-IDRI-0078


# 2. CLUSTER ANNOTATIONS(GROUPING THE DOCTOR'S DRAWINGS)

In [ ]:
import numpy as np

# Compatibility fix for older packages using np.int
np.int = int

nodules = scan.cluster_annotations()
print(f"Found {len(nodules)} distinct nodules in this scan.")

Found 4 distinct nodules in this scan.


In [ ]:
# 3. Grab the first annotation of the first nodule

In [ ]:
annotation = nodules[0][0]

print(f"Malignancy rating (1-5): {annotation.malignancy}")
print(f"Nodule Diameter: {annotation.diameter:.2f} mm")

Malignancy rating (1-5): 3
Nodule Diameter: 19.50 mm


# 4. Get the 3D numpy array of the nodule mask and the CT image bounding box

In [ ]:
annotation = nodules[0][0]
vol, mask = annotation.feature_size()
bbox = annotation.bbox()
ct_pixels = scan.to_volume()
print("Volume:", vol)
print("Bounding box:", bbox)

AttributeError: 'Annotation' object has no attribute 'feature_size'

#5. FIND THE MIDDLE SLICE OF THE 3D NODULE TO DISPLAY


In [ ]:
mid_slice = ct_pixels.shape[2]

#6. Plot the original CT and the mask overlay side by side

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,6))

In [ ]:
ax[0].imshow(ct_pixels[:, :, mid_slice], cmap='gray')
ax[0].set_title("Raw CT Scan Slice")
ax[0].axis('off')

In [ ]:
ax[1].imshow(ct_pixels[:, :, mid_slice], cmap='gray')
ax[1].imshow(mask[:, :, mid_slice], cmap='Reds', alpha=0.5) # Overlay in red
ax[1].set_title("Nodule Mask Overlay")
ax[1].axis('off')

In [ ]:
plt.tight_layout()
plt.show()